# Matching Logic for ProMatch

Find the players most similar to a user's attributes (the k-nearest-neighbours idea).

## 1. Load the players

In [1]:
import pandas as pd
import numpy as np

players = pd.read_csv("players.csv")
players.head()

,name,club,nationality,age,overall,positions,pace,shooting,passing,dribbling,defending,physical
0,L. Messi,Paris Saint-Germain,Argentina,34,93,"RW, ST, CF",85,92,91,95,34,65
1,R. Lewandowski,FC Bayern München,Poland,32,92,ST,78,92,79,86,44,82
2,Cristiano Ronaldo,Manchester United,Portugal,36,91,"ST, LW",87,94,80,88,34,75
3,Neymar Jr,Paris Saint-Germain,Brazil,29,91,"LW, CAM",91,83,86,94,37,63
4,K. De Bruyne,Manchester City,Belgium,30,91,"CM, CAM",76,86,93,88,64,78


## 2. The six attributes I match on

In [2]:
attributes = ["pace", "shooting", "passing", "dribbling", "defending", "physical"]
players[attributes].head()

,pace,shooting,passing,dribbling,defending,physical
0,85,92,91,95,34,65
1,78,92,79,86,44,82
2,87,94,80,88,34,75
3,91,83,86,94,37,63
4,76,86,93,88,64,78


## 3. Example user input

The six values a user would type into the form.

In [3]:
user = {
    "pace": 80,
    "shooting": 70,
    "passing": 75,
    "dribbling": 82,
    "defending": 40,
    "physical": 65,
}

# put the values in the same order as the attributes list
user_vector = np.array([user[a] for a in attributes])
user_vector

array([80, 70, 75, 82, 40, 65])

## 4. Euclidean distance

The overall difference between two players. Smaller = more similar.

In [4]:
def euclidean_distance(vector_a, vector_b):
    return np.sqrt(((vector_a - vector_b) ** 2).sum())

# distance between the user and every player
players["distance"] = players[attributes].apply(
    lambda row: euclidean_distance(user_vector, row.to_numpy()), axis=1
)
players[["name"] + attributes + ["distance"]].head()

,name,pace,shooting,passing,dribbling,defending,physical,distance
0,L. Messi,85,92,91,95,34,65,31.144823
1,R. Lewandowski,78,92,79,86,44,82,28.722813
2,Cristiano Ronaldo,87,94,80,88,34,75,28.670542
3,Neymar Jr,91,83,86,94,37,63,23.832751
4,K. De Bruyne,76,86,93,88,64,78,37.107951


## 5. Closest players

Sort by distance and look at the top 10.

In [5]:
closest = players.sort_values("distance").head(10)
closest[["name", "club", "positions"] + attributes + ["distance"]]

,name,club,positions,pace,shooting,passing,dribbling,defending,physical,distance
1236,R. Orsolini,Bologna,"CAM, RW",80,70,74,81,39,65,1.732051
912,Matheus Pereira,Al Hilal,"CAM, RM, RW",79,74,75,80,39,63,5.099020
1925,F. Pizzini,Defensa y Justicia,RM,78,69,72,78,40,65,5.477226
1654,R. Del Castillo,Stade Brestois 29,"RM, LM",77,68,72,79,39,65,5.656854
1501,F. Di Francesco,Empoli,"CF, LW, ST",79,70,73,81,39,60,5.656854
1618,M. Pjaca,Torino F.C.,"LW, RW, CAM",76,67,74,81,41,63,5.656854
716,Guilson Paiva,Flamengo,"RM, RW",83,69,75,78,42,68,6.244998
1589,Rúben Lameiras,Vitória de Guimarães,"RW, LW",76,71,72,79,42,64,6.324555
725,J. Harrison,Leeds United,"LM, RM",83,71,72,79,42,68,6.403124
807,A. Townsend,Everton,"RM, RW",77,74,75,79,38,67,6.480741


## 6. Distance as a match %

Turn the distance into an easy 0-100 score (smaller distance = higher %).

In [6]:
# the biggest possible distance (all six attributes 99 apart)
max_distance = np.sqrt(len(attributes) * (99 ** 2))

players["match_percent"] = ((1 - players["distance"] / max_distance) * 100).round(1)
players.sort_values("distance").head(10)[["name", "match_percent"]]

,name,match_percent
1236,R. Orsolini,99.3
912,Matheus Pereira,97.9
1925,F. Pizzini,97.7
1654,R. Del Castillo,97.7
1501,F. Di Francesco,97.7
1618,M. Pjaca,97.7
716,Guilson Paiva,97.4
1589,Rúben Lameiras,97.4
725,J. Harrison,97.4
807,A. Townsend,97.3


## 7. Cosine similarity

Measures playing style (the shape of the attributes), ignoring the overall level.

In [7]:
def cosine_similarity(vector_a, vector_b):
    dot = (vector_a * vector_b).sum()
    return dot / (np.linalg.norm(vector_a) * np.linalg.norm(vector_b))

players["style_match"] = players[attributes].apply(
    lambda row: cosine_similarity(user_vector, row.to_numpy()), axis=1
)
players.sort_values("style_match", ascending=False).head(10)[["name", "club", "positions", "style_match"]]

,name,club,positions,style_match
1236,R. Orsolini,Bologna,"CAM, RW",0.999971
1654,R. Del Castillo,Stade Brestois 29,"RM, LM",0.999909
1925,F. Pizzini,Defensa y Justicia,RM,0.999835
9055,S. Maier,Türkgücü München,"CAM, CM, LM",0.999818
5080,D. Fagundez,Austin FC,"CAM, CM, LM",0.999798
6283,Matheus Pereira,FC Barcelona,"CAM, RM",0.999797
1618,M. Pjaca,Torino F.C.,"LW, RW, CAM",0.999794
298,J. Correa,Inter,"CF, ST, CAM",0.999768
12110,L. Damer,TSV Havelse,"CAM, ST",0.999739
2687,Paulinho,Bayer 04 Leverkusen,"CAM, LW, RW",0.999713


## 8. Adding weights

Let some attributes matter more. Here pace and dribbling count double.

In [8]:
weights = {"pace": 2, "shooting": 1, "passing": 1, "dribbling": 2, "defending": 1, "physical": 1}
weight_vector = np.array([weights[a] for a in attributes])

weighted_user = user_vector * weight_vector
players["weighted_distance"] = players[attributes].apply(
    lambda row: euclidean_distance(weighted_user, row.to_numpy() * weight_vector), axis=1
)
players.sort_values("weighted_distance").head(10)[["name", "club"] + attributes]

,name,club,pace,shooting,passing,dribbling,defending,physical
1236,R. Orsolini,Bologna,80,70,74,81,39,65
1501,F. Di Francesco,Empoli,79,70,73,81,39,60
912,Matheus Pereira,Al Hilal,79,74,75,80,39,63
841,B. Traoré,Aston Villa,78,73,73,82,45,66
707,Nuno Santos,Sporting CP,81,75,75,80,43,67
1289,Trincão,Wolverhampton Wanderers,79,71,70,81,37,69
982,C. Hudson-Odoi,Chelsea,82,67,74,81,40,59
1605,T. Minamino,Liverpool,79,73,69,81,36,63
874,T. Bongonda,KRC Genk,80,71,72,82,33,61
349,A. Sánchez,Inter,78,76,78,83,43,68


## 9. One function for everything

Combines distance, cosine and weights. I will use this in the backend next.

In [9]:
def recommend(user, method="euclidean", weights=None, top_n=10):
    if weights is None:
        weights = {a: 1 for a in attributes}
    weight_vector = np.array([weights[a] for a in attributes])

    user_vector = np.array([user[a] for a in attributes]) * weight_vector

    def score(row):
        player_vector = row.to_numpy() * weight_vector
        if method == "euclidean":
            return euclidean_distance(user_vector, player_vector)
        else:
            return cosine_similarity(user_vector, player_vector)

    result = players.copy()
    result["score"] = result[attributes].apply(score, axis=1)

    # euclidean: smaller is better. cosine: bigger is better.
    ascending = (method == "euclidean")
    result = result.sort_values("score", ascending=ascending)
    return result.head(top_n)[["name", "club", "positions"] + attributes + ["score"]]


# test it
recommend(user, method="euclidean", top_n=5)

,name,club,positions,pace,shooting,passing,dribbling,defending,physical,score
1236,R. Orsolini,Bologna,"CAM, RW",80,70,74,81,39,65,1.732051
912,Matheus Pereira,Al Hilal,"CAM, RM, RW",79,74,75,80,39,63,5.099020
1925,F. Pizzini,Defensa y Justicia,RM,78,69,72,78,40,65,5.477226
1654,R. Del Castillo,Stade Brestois 29,"RM, LM",77,68,72,79,39,65,5.656854
1501,F. Di Francesco,Empoli,"CF, LW, ST",79,70,73,81,39,60,5.656854


## 10. The same thing with scikit-learn (k-NN)

My recommend() above is k-nearest-neighbours done by hand. scikit-learn has it built in, and it gives the same closest players.

In [10]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(n_neighbors=5, metric="euclidean")
knn.fit(players[attributes].to_numpy())

distances, indices = knn.kneighbors([user_vector])
players.iloc[indices[0]][["name", "club", "positions"] + attributes]

,name,club,positions,pace,shooting,passing,dribbling,defending,physical
1236,R. Orsolini,Bologna,"CAM, RW",80,70,74,81,39,65
912,Matheus Pereira,Al Hilal,"CAM, RM, RW",79,74,75,80,39,63
1925,F. Pizzini,Defensa y Justicia,RM,78,69,72,78,40,65
1501,F. Di Francesco,Empoli,"CF, LW, ST",79,70,73,81,39,60
1654,R. Del Castillo,Stade Brestois 29,"RM, LM",77,68,72,79,39,65


## 11. Quick test

A fixed set of six values, and the player names it returns.

In [11]:
test_input = {
    "pace": 85,
    "shooting": 80,
    "passing": 70,
    "dribbling": 88,
    "defending": 35,
    "physical": 70,
}

results = recommend(test_input, method="euclidean", top_n=5)
print(results["name"].tolist())
results

['A. Martial', 'Gabriel Jesus', 'Matheus Cunha', 'W. Zaha', 'D. Machís']


,name,club,positions,pace,shooting,passing,dribbling,defending,physical,score
296,A. Martial,Manchester United,"ST, LM",87,80,73,85,35,68,5.099020
168,Gabriel Jesus,Manchester City,ST,84,81,72,86,39,73,5.916080
566,Matheus Cunha,Atlético de Madrid,"CAM, LM, ST",85,78,74,84,31,69,7.280110
193,W. Zaha,Crystal Palace,"CF, LM, ST",89,77,73,87,34,75,7.810250
295,D. Machís,Granada CF,"LM, RM, ST",88,80,75,83,38,70,8.246211
